# Process All v3

Preprocess all ARC + ATLAS T1s and lesion masks to the **v3 training format**:
- ANTs Rigid+Affine+SyN registration to MNI152NLin2009cAsym (1mm)
- Intensity normalization: p1–p99 nonzero clip → z-score → rescale [0,1]
- **No skull stripping** (matches v3 training)

Output:
1. All processed images → `Processed_HiresLowres_All/t1/` and `Processed_HiresLowres_All/masks/`
2. Split exactly as v3 training: `test_hires/` (138 subjects), `test_lores/` (198 subjects), `train_hires/` (rest)

In [7]:
import sys, os, shutil
from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd

# --- ANTs & template paths ---
V4_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4")
ANTS_REG    = V4_ROOT / "tools/ants/bin/antsRegistration"
ANTS_APPLY  = V4_ROOT / "tools/ants/bin/antsApplyTransforms"
MNI_TPL     = V4_ROOT / "data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz"
MNI_TPL_2MM = V4_ROOT / "data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz"

# Point templateflow at local cache so prep_utils can find 2mm template
os.environ["TEMPLATEFLOW_HOME"] = str(V4_ROOT / "data" / "templateflow")

# Import prep_utils functions from v4
sys.path.insert(0, str(V4_ROOT / "src" / "data_prep"))
from prep_utils import ants_register, ants_apply, normalize_t1, _match_pairs, _key_from_name, _cleanup_transforms

# --- Source data paths ---
ARC_T1_DIR    = Path("/home/rbielski/ARC/combined_t1_raw")
ARC_MASK_DIR  = Path("/home/rbielski/ARC/combined_t1_raw/combined_masks_raw")
ATLAS_T1_DIR  = Path("/home/rbielski/Atlas_2/Training/Images")
ATLAS_MASK_DIR = Path("/home/rbielski/Atlas_2/Training/Masks")

# --- Output paths ---
SPLIT_BASE = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data")
ALL_OUT  = SPLIT_BASE / "Processed_HiresLowres_All"
XFM_DIR  = ALL_OUT / "_xfm_tmp"

for d in [
    ALL_OUT / "t1", ALL_OUT / "masks", XFM_DIR,
    SPLIT_BASE / "test_hires/t1",  SPLIT_BASE / "test_hires/masks",
    SPLIT_BASE / "test_lores/t1",  SPLIT_BASE / "test_lores/masks",
    SPLIT_BASE / "train_hires/t1", SPLIT_BASE / "train_hires/masks",
]:
    d.mkdir(parents=True, exist_ok=True)

# --- Processing flag ---
OVERWRITE = False  # Set True to reprocess existing outputs

# Sanity checks
for p in [ANTS_REG, ANTS_APPLY, MNI_TPL, MNI_TPL_2MM,
          ARC_T1_DIR, ARC_MASK_DIR, ATLAS_T1_DIR, ATLAS_MASK_DIR]:
    assert p.exists(), f"Not found: {p}"
# --- Parallelism ---
# ANTs is CPU-only. Limit ITK threads per job to avoid oversubscription.
# 12 jobs x 2 ITK threads = 24 threads on 16 physical / 32 logical cores.
N_JOBS = 12
ITK_THREADS = 2

from joblib import Parallel, delayed

print("All paths verified.")

All paths verified.


In [8]:
# Load exact v3 test split keys from the original evaluation manifests.
# These are fixed reference files from the v3 run — filenames are hardcoded
# to avoid accidentally picking up later files with the same prefix.
RUN_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval")

HIRES_CSV = RUN_DIR / "test_hires_metrics_with_manifest_20251111_130301.csv"
LORES_CSV = RUN_DIR / "test_lores_metrics_with_manifest_20251111_143340.csv"

for p in [HIRES_CSV, LORES_CSV]:
    assert p.exists(), f"Missing v3 manifest: {p}"

test_hires_keys = set(pd.read_csv(HIRES_CSV)["key"].tolist())
test_lores_keys = set(pd.read_csv(LORES_CSV)["key"].tolist())

print(f"test_hires: {len(test_hires_keys)} subjects  ({HIRES_CSV.name})")
print(f"test_lores: {len(test_lores_keys)} subjects  ({LORES_CSV.name})")
assert not (test_hires_keys & test_lores_keys), "ERROR: overlap between hires and lores split!"
print("No overlap confirmed.")

test_hires: 138 subjects  (test_hires_metrics_with_manifest_20251111_130301.csv)
test_lores: 198 subjects  (test_lores_metrics_with_manifest_20251111_143340.csv)
No overlap confirmed.


In [9]:
# Core per-subject processing function
def process_subject(t1: Path, mask: Path, key: str) -> bool:
    """
    Register T1 to MNI, apply transforms to T1 and mask, normalize T1.
    Returns True on success, False on error.
    """
    # Limit ANTs internal threading so parallel jobs don't oversubscribe
    os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"] = str(ITK_THREADS)

    out_t1   = ALL_OUT / "t1"    / f"{key}_T1w_MNI_norm.nii.gz"
    out_mask = ALL_OUT / "masks" / f"{key}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_t1.exists() and out_mask.exists():
        return True  # already done

    xfm_prefix = XFM_DIR / f"{key}_"
    tmp_t1     = XFM_DIR / f"{key}_t1_mni.nii.gz"
    tmp_mask   = XFM_DIR / f"{key}_mask_mni.nii.gz"

    try:
        # 1. Register T1 to MNI (Rigid + Affine + SyN), uses 2mm template internally
        ants_register(t1, xfm_prefix, MNI_TPL, ANTS_REG, use_2mm=True)

        # 2. Apply transforms to T1 (linear interpolation)
        ants_apply(t1, MNI_TPL, xfm_prefix, tmp_t1, ANTS_APPLY, nn=False)

        # 3. Apply same transforms to mask (nearest-neighbor to preserve binary)
        ants_apply(mask, MNI_TPL, xfm_prefix, tmp_mask, ANTS_APPLY, nn=True)

        # 4. Normalize T1: p1-p99 clip -> z-score -> [0,1]
        img  = nib.load(str(tmp_t1))
        norm = normalize_t1(img.get_fdata(dtype=np.float32))
        nib.save(nib.Nifti1Image(norm, img.affine, img.header), str(out_t1))

        # 5. Threshold mask to binary uint8
        mimg  = nib.load(str(tmp_mask))
        mdata = (mimg.get_fdata(dtype=np.float32) > 0.5).astype(np.uint8)
        nib.save(nib.Nifti1Image(mdata, mimg.affine, mimg.header), str(out_mask))

        # 6. Clean up intermediate files
        for f in [tmp_t1, tmp_mask]:
            if f.exists(): f.unlink()
        _cleanup_transforms(xfm_prefix)

        return True

    except Exception as e:
        print(f"  [ERROR] {key}: {e}")
        for f in [tmp_t1, tmp_mask]:
            if f.exists(): f.unlink()
        _cleanup_transforms(xfm_prefix)
        return False

print("process_subject() defined.")

process_subject() defined.


In [10]:
# Process ARC subjects (parallel)
arc_t1s   = sorted(ARC_T1_DIR.glob("sub-M*_T1w.nii.gz"))
arc_masks = sorted(ARC_MASK_DIR.glob("sub-M*mask*.nii.gz"))

arc_pairs = _match_pairs(arc_t1s, arc_masks)
print(f"\nARC: {len(arc_t1s)} T1s, {len(arc_masks)} masks -> {len(arc_pairs)} matched pairs")

# Filter to subjects not yet processed
todo_arc = [(t1, mask, _key_from_name(t1.name)) for t1, mask, _ in arc_pairs
            if OVERWRITE or not (ALL_OUT / "t1" / f"{_key_from_name(t1.name)}_T1w_MNI_norm.nii.gz").exists()]
print(f"To process: {len(todo_arc)}  (skipping {len(arc_pairs) - len(todo_arc)} already done)\n")

arc_results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
    delayed(process_subject)(t1, mask, key) for t1, mask, key in todo_arc
)

arc_ok   = sum(arc_results)
arc_fail = sum(1 for r in arc_results if not r)
arc_skip = len(arc_pairs) - len(todo_arc)
print(f"\nARC done -- processed: {arc_ok}, skipped: {arc_skip}, failed: {arc_fail}")


[match] base=0 subses=203 subject=0 missing=244 ambiguous_base=0 ambiguous_subses=0 ambiguous_subject=0
[warn] no mask for sub-M2001_ses-1076_acq-tfl3_run-4_T1w.nii.gz (key sub-M2001_ses-1076, sub sub-M2001); skipping
[warn] no mask for sub-M2002_ses-1440_acq-tfl3_run-5_T1w.nii.gz (key sub-M2002_ses-1440, sub sub-M2002); skipping
[warn] no mask for sub-M2002_ses-3770_acq-tfl3p2_run-2_T1w.nii.gz (key sub-M2002_ses-3770, sub sub-M2002); skipping
[warn] no mask for sub-M2002_ses-4006_acq-tfl3p2_run-2_T1w.nii.gz (key sub-M2002_ses-4006, sub sub-M2002); skipping
[warn] no mask for sub-M2003_ses-1408_acq-tfl3p2_run-3_T1w.nii.gz (key sub-M2003_ses-1408, sub sub-M2003); skipping
[warn] no mask for sub-M2004_ses-524_acq-tfl3p2_run-3_T1w.nii.gz (key sub-M2004_ses-524, sub sub-M2004); skipping
[warn] no mask for sub-M2005_ses-2446_acq-tfl3p2_run-2_T1w.nii.gz (key sub-M2005_ses-2446, sub sub-M2005); skipping
[warn] no mask for sub-M2005_ses-3476_acq-tfl3p2_run-11_T1w.nii.gz (key sub-M2005_ses-3476

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2012_ses-1158_acq-tfl3p2_run-3_T1w.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2012_ses-1158_acq-tfl3p2_run-3_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2012_ses-1158_acq-tfl3p2_run-3_T1w.nii.gz,1,32,Regular,0.25] -t Affine[0.1] -c 1000x500x250 -s 3x2x1vo

[Parallel(n_jobs=12)]: Done   1 tasks      | elapsed:  5.2min


>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2046_ses-2499_acq-tfl3p2_run-2_T1w.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2046_ses-2499_acq-tfl3p2_run-2_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2046_ses-2499_acq-tfl3p2_run-2_T1w.nii.gz,1,32,Regular,0.25] -t Affine[0.1] -c 1000x500x250 -s 3x2x1vo

...

[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed: 79.9min


>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2291_ses-3786_acq-tfl3p2_run-3_T1w.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2291_ses-3786_acq-tfl3p2_run-3_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/ARC/combined_t1_raw/sub-M2291_ses-3786_acq-tfl3p2_run-3_T1w.nii.gz,1,32,Regular,0.25] -t Affine[0.1] -c 1000x500x250 -s 3x2x1vo

[Parallel(n_jobs=12)]: Done 201 out of 203 | elapsed: 90.4min remaining:   54.0s


>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsApplyTransforms -d 3 -i /home/rbielski/ARC/combined_t1_raw/combined_masks_raw/sub-M2309_ses-960_acq-spc3p2_run-4_T2w_desc-lesion_mask.nii.gz -r /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz -o /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/Processed_HiresLowres_All/_xfm_tmp/sub-M2309_ses-960_mask_mni.nii.gz -n NearestNeighbor -t /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/Processed_HiresLowres_All/_xfm_tmp/sub-M2309_ses-960_1Warp.nii.gz -t /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/Processed_HiresLowres_All/_xfm_tmp/sub-M2309_ses-960_0GenericAffine.mat
>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_

[Parallel(n_jobs=12)]: Done 203 out of 203 | elapsed: 90.6min finished


In [11]:
# Process ATLAS subjects (parallel)
atlas_t1s   = sorted(ATLAS_T1_DIR.glob("sub-r*_T1w.nii.gz"))
atlas_masks = sorted(ATLAS_MASK_DIR.glob("sub-r*mask*.nii.gz"))

atlas_pairs = _match_pairs(atlas_t1s, atlas_masks)
print(f"\nATLAS: {len(atlas_t1s)} T1s, {len(atlas_masks)} masks -> {len(atlas_pairs)} matched pairs")

# Filter to subjects not yet processed
todo_atlas = [(t1, mask, _key_from_name(t1.name)) for t1, mask, _ in atlas_pairs
              if OVERWRITE or not (ALL_OUT / "t1" / f"{_key_from_name(t1.name)}_T1w_MNI_norm.nii.gz").exists()]
print(f"To process: {len(todo_atlas)}  (skipping {len(atlas_pairs) - len(todo_atlas)} already done)\n")

atlas_results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
    delayed(process_subject)(t1, mask, key) for t1, mask, key in todo_atlas
)

atlas_ok   = sum(atlas_results)
atlas_fail = sum(1 for r in atlas_results if not r)
atlas_skip = len(atlas_pairs) - len(todo_atlas)
print(f"\nATLAS done -- processed: {atlas_ok}, skipped: {atlas_skip}, failed: {atlas_fail}")


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


[match] base=0 subses=655 subject=0 missing=0 ambiguous_base=0 ambiguous_subses=0 ambiguous_subject=0

ATLAS: 655 T1s, 655 masks -> 655 matched pairs
To process: 655  (skipping 0 already done)

>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r001s001_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r001s001_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/t

[Parallel(n_jobs=12)]: Done   1 tasks      | elapsed:  5.1min


>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsApplyTransforms -d 3 -i /home/rbielski/Atlas_2/Training/Images/sub-r001s002_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz -r /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz -o /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/Processed_HiresLowres_All/_xfm_tmp/sub-r001s002_ses-1_t1_mni.nii.gz -t /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/Processed_HiresLowres_All/_xfm_tmp/sub-r001s002_ses-1_1Warp.nii.gz -t /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/Processed_HiresLowres_All/_xfm_tmp/sub-r001s002_ses-1_0GenericAffine.mat
>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsRegistration -d 3 -

[Parallel(n_jobs=12)]: Done   8 tasks      | elapsed:  5.3min


...

[Parallel(n_jobs=12)]: Done 589 tasks      | elapsed: 262.9min


>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r048s010_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r048s010_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r048s010_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1,32,Regular,0.25] -t Af

[Parallel(n_jobs=12)]: Done 624 tasks      | elapsed: 274.9min


>> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r050s008_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r050s008_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r050s008_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1,32,Regular,0.25] -t Af

[Parallel(n_jobs=12)]: Done 655 out of 655 | elapsed: 288.7min finished


In [2]:
# Apply the exact v3 split: copy files (not symlinks) to test_hires, test_lores, train_hires
import shutil, pandas as pd
from pathlib import Path

SPLIT_BASE = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data")
ALL_OUT    = SPLIT_BASE / "Processed_HiresLowres_All"
OVERWRITE  = False

RUN_DIR    = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/test_eval")
HIRES_CSV  = RUN_DIR / "test_hires_metrics_with_manifest_20251111_130301.csv"
LORES_CSV  = RUN_DIR / "test_lores_metrics_with_manifest_20251111_143340.csv"
test_hires_keys = set(pd.read_csv(HIRES_CSV)["key"].tolist())
test_lores_keys = set(pd.read_csv(LORES_CSV)["key"].tolist())


processed_t1s = sorted((ALL_OUT / "t1").glob("*_T1w_MNI_norm.nii.gz"))
print(f"Total processed T1s available: {len(processed_t1s)}")

counts = {"test_hires": 0, "test_lores": 0, "train_hires": 0, "no_mask": 0}
no_mask_keys = []

for t1_path in processed_t1s:
    key = t1_path.name.replace("_T1w_MNI_norm.nii.gz", "")
    mask_path = ALL_OUT / "masks" / f"{key}_lesion_mask_MNI_clean.nii.gz"

    if not mask_path.exists():
        print(f"[warn] no mask for {key}, skipping split")
        counts["no_mask"] += 1
        no_mask_keys.append(key)
        continue

    if key in test_hires_keys:
        split = "test_hires"
    elif key in test_lores_keys:
        split = "test_lores"
    else:
        split = "train_hires"

    dst_t1   = SPLIT_BASE / split / "t1"    / t1_path.name
    dst_mask = SPLIT_BASE / split / "masks" / mask_path.name

    if not OVERWRITE and dst_t1.exists() and dst_mask.exists():
        continue

    # Remove broken symlinks (from old A_A_Combined_Data) before copying real files
    for p in [dst_t1, dst_mask]:
        if p.is_symlink():
            p.unlink()

    shutil.copy2(str(t1_path),   str(dst_t1))
    shutil.copy2(str(mask_path), str(dst_mask))
    counts[split] += 1

print(f"\nSplit results (newly copied):")
for k, v in counts.items():
    print(f"  {k}: {v}")

Total processed T1s available: 858

Split results (newly copied):
  test_hires: 138
  test_lores: 198
  train_hires: 521
  no_mask: 0


In [3]:
# Verification summary
print("=" * 60)
print("VERIFICATION SUMMARY")
print("=" * 60)

for split in ["test_hires", "test_lores", "train_hires"]:
    t1s   = list((SPLIT_BASE / split / "t1").glob("*.nii.gz"))
    masks = list((SPLIT_BASE / split / "masks").glob("*.nii.gz"))
    print(f"{split:15s}: {len(t1s):4d} T1s, {len(masks):4d} masks")

print()

hires_found_keys = {f.name.replace("_T1w_MNI_norm.nii.gz", "")
                    for f in (SPLIT_BASE / "test_hires" / "t1").glob("*.nii.gz")}
lores_found_keys = {f.name.replace("_T1w_MNI_norm.nii.gz", "")
                    for f in (SPLIT_BASE / "test_lores" / "t1").glob("*.nii.gz")}

hires_missing = test_hires_keys - hires_found_keys
lores_missing = test_lores_keys - lores_found_keys

if hires_missing:
    print(f"WARNING: {len(hires_missing)} test_hires subjects missing from output:")
    for k in sorted(hires_missing): print(f"  {k}")
else:
    print(f"test_hires: all {len(test_hires_keys)} expected subjects present.")

if lores_missing:
    print(f"WARNING: {len(lores_missing)} test_lores subjects missing from output:")
    for k in sorted(lores_missing): print(f"  {k}")
else:
    print(f"test_lores: all {len(test_lores_keys)} expected subjects present.")

print()
all_t1s   = list((ALL_OUT / "t1").glob("*.nii.gz"))
all_masks = list((ALL_OUT / "masks").glob("*.nii.gz"))
print(f"Processed_HiresLowres_All: {len(all_t1s)} T1s, {len(all_masks)} masks")

VERIFICATION SUMMARY
test_hires     :  314 T1s,  314 masks
test_lores     :  302 T1s,  302 masks
train_hires    :  662 T1s,  662 masks

test_hires: all 138 expected subjects present.
test_lores: all 198 expected subjects present.

Processed_HiresLowres_All: 858 T1s, 858 masks
